# 指甲分割 (NailSeg) — PointNeXt Kaggle 训练笔记本

本笔记本提供了在 Kaggle 上使用 **PointNeXt** 系列模型进行指甲点云分割的完整工作流。

**支持的模型：**
- PointNet
- PointNet++
- PointNeXt-S

**任务说明：**
- **输入**：点云 (XYZ + 高度特征)，每个文件为 `.npy` 格式
- **类别**：2 类（非指甲 / 指甲）
- **评估指标**：mIoU、OA (整体精度)、mAcc (平均精度)

**参考论文：** [PointNeXt: Revisiting PointNet++ with Improved Training and Scaling Strategies](https://arxiv.org/abs/2206.04670) (NeurIPS 2022)

> **Kaggle 设置：**
> - 在 *Settings → Accelerator* 中启用 **GPU T4 x2**（或 P100）
> - 在 *Settings → Internet* 中启用 **Internet → On**
> - 将数据集上传为 Kaggle Dataset，避免重复下载

## 1. 环境配置

安装 PointNeXt 所需的全部依赖并编译 CUDA 扩展。

### 1.1 克隆代码库

In [ ]:
import os

REPO_DIR = '/kaggle/working/PointNeXt'
if not os.path.exists(REPO_DIR):
    !git clone --recurse-submodules https://github.com/frank2033/PointNeXt.git {REPO_DIR}
os.chdir(REPO_DIR)
print(f'工作目录: {os.getcwd()}')

### 1.2 安装 Python 依赖

In [ ]:
import torch

# 安装核心依赖
!pip install -q torch-scatter -f https://data.pyg.org/whl/torch-{torch.__version__.split('+')[0]}+cu118.html
!pip install -q ninja easydict==1.9 pyyaml==6.0 h5py==3.6.0 scikit-learn==1.0.2 \
    tensorboard==2.8.0 tqdm wandb pyvista pandas Cython shortuuid multimethod

### 1.3 编译 CUDA 扩展

编译 PointNet++ CUDA 核心操作（必须）。

In [ ]:
# 编译 PointNet++ batch CUDA 操作（必须）
os.chdir(os.path.join(REPO_DIR, 'openpoints', 'cpp', 'pointnet2_batch'))
!python setup.py install --user 2>&1 | tail -5

# 返回项目根目录
os.chdir(REPO_DIR)
print('CUDA 扩展编译成功。')

### 1.4 验证安装

In [ ]:
import torch
print(f'PyTorch 版本: {torch.__version__}')
print(f'CUDA 可用: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA 设备: {torch.cuda.get_device_name(0)}')
    print(f'CUDA 版本: {torch.version.cuda}')

import sys
sys.path.insert(0, REPO_DIR)
from openpoints.utils import EasyConfig
print('\nOpenPoints 库加载成功！')

## 2. 数据准备

数据集位于 `/kaggle/input/pointcloud/npy_files`，包含 `.npy` 格式的点云文件。

每个 `.npy` 文件包含形状为 `(N, C)` 的 numpy 数组：
- 第 0-2 列：x, y, z 坐标
- 第 3 到 C-2 列（可选）：附加特征（如法线、颜色）
- 最后一列 (C-1)：标签（0=非指甲，1=指甲）

支持两种数据目录结构：

**方式一：已划分好的数据集（推荐）**
```
npy_files/
    train/*.npy
    val/*.npy
    test/*.npy
```
如果数据源中已存在 `train/`、`val/`、`test/` 子目录，将直接使用。

**方式二：未划分的数据集**
```
npy_files/*.npy
```
如果数据源是扁平目录（所有 `.npy` 文件在同一级），将自动按 **8:1:1** 划分。

最终目录结构：
```
data/NailSeg/
    train/*.npy
    val/*.npy
    test/*.npy
```

In [ ]:
import os
import glob
import numpy as np

os.chdir(REPO_DIR)

# ===== 数据集路径（修改此处指向你的数据）=====
DATA_SOURCE = '/kaggle/input/pointcloud/npy_files'

# 目标目录
DATA_ROOT = os.path.join(REPO_DIR, 'data', 'NailSeg')

# 检测数据源是否已包含 train/val/test 子目录
has_train = os.path.isdir(os.path.join(DATA_SOURCE, 'train'))
has_val = os.path.isdir(os.path.join(DATA_SOURCE, 'val'))
has_test = os.path.isdir(os.path.join(DATA_SOURCE, 'test'))
pre_split = has_train and has_val and has_test

if pre_split:
    # ===== 方式一：数据已划分好，直接使用 =====
    print('检测到已划分的数据集目录结构（train/val/test），跳过自动划分。')
    for split in ['train', 'val', 'test']:
        src_dir = os.path.join(DATA_SOURCE, split)
        dst_dir = os.path.join(DATA_ROOT, split)
        os.makedirs(dst_dir, exist_ok=True)
        npy_files = sorted(glob.glob(os.path.join(src_dir, '*.npy')))
        for src_path in npy_files:
            dst_path = os.path.join(dst_dir, os.path.basename(src_path))
            if not os.path.exists(dst_path):
                os.symlink(src_path, dst_path)
else:
    # ===== 方式二：扁平目录，自动按 8:1:1 划分 =====
    from sklearn.model_selection import train_test_split
    print('未检测到 train/val/test 子目录，自动按 8:1:1 划分数据。')

    all_npy = sorted(glob.glob(os.path.join(DATA_SOURCE, '*.npy')))
    if len(all_npy) == 0:
        raise FileNotFoundError(f'在 {DATA_SOURCE} 中未找到 .npy 文件。请检查路径。')

    # 先分出 80% train + 20% temp，再将 temp 对半分为 val 和 test（各 10%）
    train_files, temp_files = train_test_split(all_npy, test_size=0.2, random_state=42)
    val_files, test_files = train_test_split(temp_files, test_size=0.5, random_state=42)

    for split, files in [('train', train_files), ('val', val_files), ('test', test_files)]:
        split_dir = os.path.join(DATA_ROOT, split)
        os.makedirs(split_dir, exist_ok=True)
        for src_path in files:
            dst_path = os.path.join(split_dir, os.path.basename(src_path))
            if not os.path.exists(dst_path):
                os.symlink(src_path, dst_path)

# 查看数据格式（从训练集取一个样本）
train_dir = os.path.join(DATA_ROOT, 'train')
sample_files = sorted(glob.glob(os.path.join(train_dir, '*.npy')))
if len(sample_files) > 0:
    sample = np.load(sample_files[0])
    print(f'\n样本数据格式: shape={sample.shape}, dtype={sample.dtype}')
    print(f'  列数: {sample.shape[1]} (前3列=XYZ, 最后1列=标签)')
    labels = sample[:, -1].astype(int)
    print(f'  标签值: {np.unique(labels)} (0=非指甲, 1=指甲)')

# 验证
print(f'\n数据目录: {DATA_ROOT}')
for split in ['train', 'val', 'test']:
    split_dir = os.path.join(DATA_ROOT, split)
    npy_count = len([f for f in os.listdir(split_dir) if f.endswith('.npy')]) if os.path.isdir(split_dir) else 0
    print(f'  {split}: {npy_count} 个文件')

## 3. 辅助函数

定义配置加载、训练、验证和可视化的通用函数。

In [ ]:
import yaml
import logging
import numpy as np
import json
import torch
import torch.nn as nn
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm
import matplotlib.pyplot as plt

from openpoints.utils import (
    EasyConfig, dist_utils, set_random_seed, save_checkpoint,
    load_checkpoint, resume_checkpoint, setup_logger_dist,
    cal_model_parm_nums, Wandb, generate_exp_directory,
    resume_exp_directory, AverageMeter, ConfusionMatrix, get_mious
)
from openpoints.dataset import build_dataloader_from_cfg, get_features_by_keys, get_class_weights
from openpoints.transforms import build_transforms_from_cfg
from openpoints.optim import build_optimizer_from_cfg
from openpoints.scheduler import build_scheduler_from_cfg
from openpoints.loss import build_criterion_from_cfg
from openpoints.models import build_model_from_cfg


def setup_cfg_for_notebook(cfg_path, overrides=None):
    """
    加载 PointNeXt 配置文件，适配 Kaggle 单 GPU 训练。

    Args:
        cfg_path: YAML 配置文件路径。
        overrides: 配置覆盖字典，例如 {'epochs': 50, 'batch_size': 16}。

    Returns:
        配置好的 EasyConfig 对象。
    """
    cfg = EasyConfig()
    cfg.load(cfg_path, recursive=True)

    if overrides:
        for k, v in overrides.items():
            cfg[k] = v

    if cfg.seed is None:
        cfg.seed = np.random.randint(1, 10000)

    # 单 GPU 设置（不使用分布式）
    cfg.rank = 0
    cfg.world_size = 1
    cfg.distributed = False
    cfg.mp = False
    cfg.sync_bn = False

    # 默认关闭 wandb（如有 API key 可设为 True）
    cfg.wandb.use_wandb = False

    # 生成实验目录
    cfg.task_name = cfg_path.split('.')[-2].split('/')[-2]
    cfg.cfg_basename = cfg_path.split('.')[-2].split('/')[-1]
    cfg.exp_name = cfg.cfg_basename
    tags = [cfg.task_name, cfg.mode, cfg.cfg_basename,
            f'ngpus{cfg.world_size}', f'seed{cfg.seed}']
    cfg.opts = ''

    cfg.root_dir = os.path.join(cfg.root_dir, cfg.task_name)
    cfg.is_training = cfg.mode not in ['test', 'testing', 'val', 'eval', 'evaluation']
    generate_exp_directory(cfg, tags, additional_id=None)

    os.environ['JOB_LOG_DIR'] = cfg.log_dir
    cfg.cfg_path = os.path.join(cfg.run_dir, 'cfg.yaml')
    with open(cfg.cfg_path, 'w') as f:
        yaml.dump(cfg, f, indent=2)

    return cfg


def train_one_epoch(model, train_loader, criterion, optimizer, scheduler, epoch, cfg):
    """训练一个 epoch。"""
    loss_meter = AverageMeter()
    cm = ConfusionMatrix(num_classes=cfg.num_classes, ignore_index=cfg.ignore_index)
    model.train()
    pbar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f'Train E{epoch}')
    num_iter = 0
    for idx, data in pbar:
        for key in data.keys():
            data[key] = data[key].cuda(non_blocking=True)
        num_iter += 1
        target = data['y']
        data['x'] = get_features_by_keys(data, cfg.feature_keys)
        logits = model(data)
        loss = criterion(logits, target)
        loss.backward()
        if num_iter == cfg.step_per_update:
            if cfg.get('grad_norm_clip') is not None and cfg.grad_norm_clip > 0.:
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_norm_clip, norm_type=2)
            num_iter = 0
            optimizer.step()
            optimizer.zero_grad()
            if not cfg.sched_on_epoch:
                scheduler.step(epoch)
        cm.update(logits.argmax(dim=1), target)
        loss_meter.update(loss.item())
        if idx % cfg.print_freq == 0:
            pbar.set_description(f'Train E[{epoch}/{cfg.epochs}] Loss {loss_meter.val:.3f} Acc {cm.overall_accuray:.2f}')
    return loss_meter.avg


@torch.no_grad()
def validate(model, val_loader, cfg):
    """验证模型。"""
    model.eval()
    cm = ConfusionMatrix(num_classes=cfg.num_classes, ignore_index=cfg.ignore_index)
    for idx, data in tqdm(enumerate(val_loader), total=len(val_loader), desc='Val'):
        for key in data.keys():
            data[key] = data[key].cuda(non_blocking=True)
        target = data['y']
        data['x'] = get_features_by_keys(data, cfg.feature_keys)
        logits = model(data)
        cm.update(logits.argmax(dim=1), target)
    tp, union, count = cm.tp, cm.union, cm.count
    miou, macc, oa, ious, accs = get_mious(tp, union, count)
    return miou, macc, oa, ious, accs


def plot_training_curves(history, title='训练曲线'):
    """绘制训练损失和指标曲线。"""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    epochs = history['epochs']
    axes[0].plot(epochs, history['train_loss'], label='Train Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title(f'{title} - 损失')
    axes[0].legend()
    axes[0].grid(True)

    for key in history:
        if key not in ('epochs', 'train_loss', 'lr') and len(history[key]) == len(epochs):
            axes[1].plot(epochs, history[key], label=key)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('指标值')
    axes[1].set_title(f'{title} - 评估指标')
    axes[1].legend()
    axes[1].grid(True)

    plt.tight_layout()
    plt.show()


def plot_comparison(results_dict, title='模型对比'):
    """绘制多个模型的对比曲线。"""
    fig, axes = plt.subplots(1, 3, figsize=(20, 5))
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

    for i, (name, history) in enumerate(results_dict.items()):
        c = colors[i % len(colors)]
        epochs = history['epochs']
        axes[0].plot(epochs, history['train_loss'], color=c, linewidth=2, label=name)
        axes[1].plot(epochs, history['val_oa'], color=c, linewidth=2, label=name)
        axes[2].plot(epochs, history['val_miou'], color=c, linewidth=2, label=name)

    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].set_title('训练损失'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('OA (%)')
    axes[1].set_title('验证精度 (OA)'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
    axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('mIoU (%)')
    axes[2].set_title('验证 mIoU'); axes[2].legend(); axes[2].grid(True, alpha=0.3)

    fig.suptitle(title, fontsize=16, y=1.02)
    plt.tight_layout()
    plt.show()


print('辅助函数定义完成。')

## 4. 注册 NailSeg 数据集

导入并注册自定义数据集类。

In [ ]:
import sys
sys.path.insert(0, os.path.join(REPO_DIR, 'examples', 'nailseg'))
from nailseg_dataset import NailSeg

print(f'NailSeg 数据集已注册。')
print(f'  类别数: {NailSeg.num_classes}')
print(f'  类别名: {NailSeg.classes}')

### 4.1 自定义数据集类（可选）

除了 openpoints 兼容的 `NailSeg` 类，项目还提供了独立的 `CustomDataset` 类。
该类可脱离 openpoints 框架独立使用，返回 `(points, seg)` 元组格式。

> **注意**：本笔记本默认使用 `NailSeg` 类（与 openpoints 训练流程兼容）。
> 如需在其他项目中使用 `CustomDataset`，可参考 `examples/nailseg/custom_dataset.py`。

In [ ]:
# 导入独立的 CustomDataset（可选，用于脱离 openpoints 框架的场景）
from custom_dataset import CustomDataset, pc_normalize

# 快速验证 CustomDataset 可加载数据
try:
    _test_ds = CustomDataset(root=DATA_ROOT, split='train', num_points=1024,
                             data_augmentation=False)
    _pts, _seg = _test_ds[0]
    print(f'CustomDataset 验证通过:')
    print(f'  样本数: {len(_test_ds)}')
    print(f'  点云形状: {_pts.shape} (特征维度={_pts.shape[1]})')
    print(f'  标签形状: {_seg.shape}, 标签值: {_seg.unique().tolist()}')
    del _test_ds, _pts, _seg
except Exception as e:
    print(f'CustomDataset 验证跳过: {e}')

## 5. 选择模型与配置

选择要训练的模型。可选：
- `pointnet.yaml` — PointNet（基线）
- `pointnet++.yaml` — PointNet++
- `pointnext-s.yaml` — PointNeXt-S（推荐）

> 修改 `MODEL_CONFIG` 变量来切换模型。

In [ ]:
# ===== 选择模型（修改这里切换模型）=====
MODEL_CONFIG = 'pointnext-s.yaml'  # 可选: 'pointnet.yaml', 'pointnet++.yaml', 'pointnext-s.yaml'

# ===== 训练参数（可根据需要调整）=====
EPOCHS = 100          # 完整训练建议 200
BATCH_SIZE = 16       # 如遇 OOM 错误，减小此值
VAL_FREQ = 5          # 每隔多少 epoch 验证一次
NUM_WORKERS = 2       # 数据加载线程数
LEARNING_RATE = 0.001 # 学习率

nail_cfg_path = os.path.join(REPO_DIR, 'cfgs', 'nailseg', MODEL_CONFIG)
nail_cfg = setup_cfg_for_notebook(nail_cfg_path, overrides={
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'val_freq': VAL_FREQ,
    'lr': LEARNING_RATE,
    'dataloader': {'num_workers': NUM_WORKERS},
})

# 覆盖数据路径为我们准备好的目录
nail_cfg.dataset.common.data_root = DATA_ROOT

setup_logger_dist(nail_cfg.log_path, nail_cfg.rank, name=nail_cfg.dataset.common.NAME)
set_random_seed(nail_cfg.seed, deterministic=nail_cfg.deterministic)

print(f'配置文件: {nail_cfg_path}')
print(f'数据路径: {nail_cfg.dataset.common.data_root}')
print(f'模型: {nail_cfg.model.NAME}')
print(f'编码器: {nail_cfg.model.encoder_args.NAME}')
print(f'类别数: {nail_cfg.num_classes}')
print(f'训练轮数: {nail_cfg.epochs}')
print(f'批大小: {nail_cfg.batch_size}')
print(f'学习率: {nail_cfg.lr}')

## 6. 构建模型与数据加载器

In [ ]:
# 构建模型
if nail_cfg.model.get('in_channels', None) is None:
    nail_cfg.model.in_channels = nail_cfg.model.encoder_args.in_channels
nail_model = build_model_from_cfg(nail_cfg.model).cuda()
model_size = cal_model_parm_nums(nail_model)
print(f'模型参数量: {model_size / 1e6:.4f} M')

# 构建数据加载器
try:
    nail_train_loader = build_dataloader_from_cfg(
        nail_cfg.batch_size, nail_cfg.dataset, nail_cfg.dataloader,
        datatransforms_cfg=nail_cfg.datatransforms, split='train', distributed=False)
    nail_val_loader = build_dataloader_from_cfg(
        nail_cfg.get('val_batch_size', nail_cfg.batch_size), nail_cfg.dataset, nail_cfg.dataloader,
        datatransforms_cfg=nail_cfg.datatransforms, split='val', distributed=False)
    nail_cfg.classes = nail_val_loader.dataset.classes if hasattr(
        nail_val_loader.dataset, 'classes') else [str(i) for i in range(nail_cfg.num_classes)]

    print(f'训练集样本数: {len(nail_train_loader.dataset)}')
    print(f'验证集样本数: {len(nail_val_loader.dataset)}')
    nail_data_ready = True
except Exception as e:
    print(f'数据加载失败: {e}')
    print('请确保数据已放置在 data/NailSeg/{{train,val,test}}/ 目录下。')
    nail_data_ready = False

### 6.1 数据样本可视化

查看数据格式和点云示例。

In [ ]:
if nail_data_ready:
    # 获取一个样本
    sample = nail_train_loader.dataset[0]
    print('样本键值:', list(sample.keys()))
    for k, v in sample.items():
        if hasattr(v, 'shape'):
            print(f'  {k}: shape={v.shape}, dtype={v.dtype}')
        else:
            print(f'  {k}: {type(v)}')

    # 可视化点云（2D 投影）
    pos = sample['pos'] if isinstance(sample['pos'], np.ndarray) else sample['pos'].numpy()
    label = sample['y'] if isinstance(sample['y'], np.ndarray) else sample['y'].numpy()

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    titles = ['XY 平面', 'XZ 平面', 'YZ 平面']
    pairs = [(0, 1), (0, 2), (1, 2)]
    axis_labels = [('X', 'Y'), ('X', 'Z'), ('Y', 'Z')]

    for ax, (i, j), title, (xl, yl) in zip(axes, pairs, titles, axis_labels):
        scatter = ax.scatter(pos[:, i], pos[:, j], c=label, cmap='coolwarm',
                            s=2, alpha=0.8)
        ax.set_xlabel(xl)
        ax.set_ylabel(yl)
        ax.set_title(title)
        ax.set_aspect('equal')
    plt.colorbar(scatter, ax=axes[-1], label='类别 (0=非指甲, 1=指甲)')
    plt.suptitle('点云样本可视化', fontsize=14)
    plt.tight_layout()
    plt.show()

    # 类别分布
    nail_count = np.sum(label == 1)
    total = len(label)
    print(f'\n类别分布 (当前样本):')
    print(f'  非指甲: {total - nail_count} 点 ({(total - nail_count) / total * 100:.1f}%)')
    print(f'  指甲:   {nail_count} 点 ({nail_count / total * 100:.1f}%)')

## 7. 模型训练

开始训练模型。训练过程中会记录损失和评估指标。

In [ ]:
if nail_data_ready:
    # 构建优化器、调度器和损失函数
    nail_optimizer = build_optimizer_from_cfg(nail_model, lr=nail_cfg.lr, **nail_cfg.optimizer)
    nail_scheduler = build_scheduler_from_cfg(nail_cfg, nail_optimizer)

    nail_cfg.criterion_args.weight = None
    if nail_cfg.get('cls_weighed_loss', False):
        if hasattr(nail_train_loader.dataset, 'num_per_class'):
            nail_cfg.criterion_args.weight = get_class_weights(
                nail_train_loader.dataset.num_per_class, normalize=True)
    nail_criterion = build_criterion_from_cfg(nail_cfg.criterion_args).cuda()

    # 训练指标记录
    nail_history = {
        'epochs': [], 'train_loss': [], 'lr': [],
        'val_miou': [], 'val_macc': [], 'val_oa': []
    }
    best_miou, best_macc, best_oa, best_epoch = 0., 0., 0., 0

    print(f'开始训练 {nail_cfg.model.encoder_args.NAME}...')
    print(f'总轮数: {nail_cfg.epochs}, 批大小: {nail_cfg.batch_size}')
    print('=' * 60)

    for epoch in range(nail_cfg.start_epoch, nail_cfg.epochs + 1):
        if hasattr(nail_train_loader.dataset, 'epoch'):
            nail_train_loader.dataset.epoch = epoch - 1
        nail_cfg.epoch = epoch

        train_loss = train_one_epoch(
            nail_model, nail_train_loader, nail_criterion,
            nail_optimizer, nail_scheduler, epoch, nail_cfg)

        val_miou, val_macc, val_oa = 0., 0., 0.
        if epoch % nail_cfg.val_freq == 0:
            val_miou, val_macc, val_oa, val_ious, _ = validate(
                nail_model, nail_val_loader, nail_cfg)
            is_best = val_miou > best_miou
            if is_best:
                best_miou = val_miou
                best_macc = val_macc
                best_oa = val_oa
                best_epoch = epoch
                print(f'\n★ 新最优 @E{epoch}: mIoU={val_miou:.2f}, OA={val_oa:.2f}, mAcc={val_macc:.2f}')
            save_checkpoint(nail_cfg, nail_model, epoch, nail_optimizer, nail_scheduler,
                            additioanl_dict={'best_val': best_miou}, is_best=is_best)

        lr = nail_optimizer.param_groups[0]['lr']
        if nail_cfg.sched_on_epoch:
            nail_scheduler.step(epoch)

        nail_history['epochs'].append(epoch)
        nail_history['train_loss'].append(train_loss)
        nail_history['lr'].append(lr)
        nail_history['val_miou'].append(float(val_miou))
        nail_history['val_macc'].append(float(val_macc))
        nail_history['val_oa'].append(float(val_oa))

    print('\n' + '=' * 60)
    print(f'训练完成！最优结果: mIoU={best_miou:.2f}, OA={best_oa:.2f}, mAcc={best_macc:.2f} @Epoch {best_epoch}')
else:
    print('跳过训练 — 数据不可用。请先准备数据。')

### 7.1 训练曲线可视化

In [ ]:
if nail_data_ready:
    plot_training_curves(nail_history, title=f'NailSeg ({MODEL_CONFIG.replace(".yaml", "")})')

## 8. 评估最优模型

加载训练过程中保存的最优检查点，在验证集上进行最终评估。

In [ ]:
if nail_data_ready:
    # 加载最优模型
    best_ckpt_path = os.path.join(nail_cfg.ckpt_dir, f'{nail_cfg.run_name}_ckpt_best.pth')
    if os.path.exists(best_ckpt_path):
        load_checkpoint(nail_model, pretrained_path=best_ckpt_path)
        print(f'已加载最优检查点: {best_ckpt_path}')

        test_miou, test_macc, test_oa, test_ious, test_accs = validate(
            nail_model, nail_val_loader, nail_cfg)

        print(f'\n{"="*50}')
        print(f'最终评估结果:')
        print(f'  OA    (整体精度):  {test_oa:.2f}%')
        print(f'  mAcc  (平均精度):  {test_macc:.2f}%')
        print(f'  mIoU  (平均IoU):   {test_miou:.2f}%')
        print(f'\n各类别 IoU:')
        for cls_name, iou in zip(nail_cfg.classes, test_ious):
            print(f'  {cls_name}: {iou:.2f}%')
        print(f'{"="*50}')
    else:
        print(f'未找到最优检查点: {best_ckpt_path}')

## 9. 模型对比（可选）

训练多个模型后，对比它们的性能。每次训练不同模型后，将 `nail_history` 保存到 `all_results` 字典中。

> **使用方法**：修改第 5 节的 `MODEL_CONFIG`，重新运行第 5-7 节，然后运行此节。

In [ ]:
# 将当前模型的结果保存到对比字典（每次训练后运行此单元格）
if nail_data_ready:
    if 'all_results' not in dir():
        all_results = {}
    model_name = MODEL_CONFIG.replace('.yaml', '')
    all_results[model_name] = nail_history.copy()
    print(f'已保存 {model_name} 的结果。当前已有模型: {list(all_results.keys())}')

In [ ]:
# 绘制模型对比图（需要至少 2 个模型的结果）
if 'all_results' in dir() and len(all_results) >= 2:
    plot_comparison(all_results, title='NailSeg 模型对比')

    # 对比表格
    print(f'\n{"模型":<15} {"mIoU":<10} {"OA":<10} {"mAcc":<10}')
    print('-' * 45)
    for name, hist in all_results.items():
        best_idx = np.argmax(hist['val_miou'])
        print(f'{name:<15} {hist["val_miou"][best_idx]:<10.2f} '
              f'{hist["val_oa"][best_idx]:<10.2f} {hist["val_macc"][best_idx]:<10.2f}')
elif 'all_results' in dir():
    print(f'当前只有 {len(all_results)} 个模型结果。请训练更多模型后再运行此单元格。')
else:
    print('暂无模型结果。请先运行训练。')

## 10. 保存结果与下载

将最优检查点和训练曲线保存到 Kaggle 输出目录。

In [ ]:
import shutil

output_dir = '/kaggle/working/results'
os.makedirs(output_dir, exist_ok=True)

# 复制最优检查点
log_root = os.path.join(REPO_DIR, 'log')
if os.path.exists(log_root):
    for root, dirs, files in os.walk(log_root):
        for f in files:
            if 'best' in f and f.endswith('.pth'):
                src = os.path.join(root, f)
                dst = os.path.join(output_dir, f)
                shutil.copy2(src, dst)
                print(f'已保存检查点: {dst}')

# 保存训练指标
if nail_data_ready:
    metrics_path = os.path.join(output_dir, 'metrics.json')
    with open(metrics_path, 'w') as f:
        json.dump(nail_history, f, indent=2)
    print(f'训练指标已保存: {metrics_path}')

print(f'\n所有结果已保存到 {output_dir}')
print('笔记本运行完成后，在 Kaggle 的 Output 标签页下载结果。')

## 11. Kaggle 训练技巧

### 内存管理
- Kaggle T4 GPU 有 **15 GB** 显存。如遇 OOM 错误，减小 `BATCH_SIZE`。
- 可以减小 `num_points` 以降低内存消耗。

### 训练时间
- Kaggle 每次会话限制 **12 小时**（GPU 模式），请合理设置 epoch 数量。
- 使用 `mode=resume` 和 `pretrained_path` 可跨会话继续训练。

### 数据管理
- 将预处理数据上传为 **Kaggle Dataset**，避免重复下载。
- 使用符号链接：`!ln -s /kaggle/input/<数据集名称> data/NailSeg`

### 保存检查点
- 检查点自动保存到 `log/nailseg/` 目录。
- 使用 `Save & Run All` 运行后从 Output 标签页下载。
- 对于长时间训练，可将中间检查点保存为 Kaggle Dataset。

### 恢复训练
```python
# 在第 5 节的 setup_cfg_for_notebook 后添加：
nail_cfg.mode = 'resume'
nail_cfg.pretrained_path = '/kaggle/input/my-checkpoints/ckpt_best.pth'
resume_checkpoint(nail_cfg, nail_model, nail_optimizer, nail_scheduler,
                  pretrained_path=nail_cfg.pretrained_path)
```

### 配置调整
常用配置覆盖项：
- `EPOCHS`: 训练轮数
- `BATCH_SIZE`: 批大小
- `LEARNING_RATE`: 学习率
- `VAL_FREQ`: 验证频率
- `NUM_WORKERS`: 数据加载线程数